# 6. Interactive Launch-Site Map with Folium
The map displays launch site locations and clusters launch markers. For an exact match to the course map, use the course's `spacex_launch_geo.csv` file.

In [1]:
import pandas as pd
import folium
from folium.plugins import MarkerCluster
from IPython.display import display

df = pd.read_csv("https://raw.githubusercontent.com/chuksoo/IBM-Data-Science-Capstone-SpaceX/main/spacex_launch_geo.csv")
print("Shape:", df.shape)
print(df.columns.tolist())
display(df.head())

Shape: (56, 14)
['Flight Number', 'Date', 'Time (UTC)', 'Booster Version', 'Launch Site', 'Payload', 'Payload Mass (kg)', 'Orbit', 'Customer', 'Mission Outcome', 'Landing Outcome', 'class', 'Lat', 'Long']


,Flight Number,Date,Time (UTC),Booster Version,Launch Site,Payload,Payload Mass (kg),Orbit,Customer,Mission Outcome,Landing Outcome,class,Lat,Long
0,1,2010-06-04,18:45:00,F9 v1.0 B0003,CCAFS LC-40,Dragon Spacecraft Qualification Unit,0.0,LEO,SpaceX,Success,Failure (parachute),0,28.562302,-80.577356
1,2,2010-12-08,15:43:00,F9 v1.0 B0004,CCAFS LC-40,"Dragon demo flight C1, two CubeSats, barrel o...",0.0,LEO (ISS),NASA (COTS) NRO,Success,Failure (parachute),0,28.562302,-80.577356
2,3,2012-05-22,7:44:00,F9 v1.0 B0005,CCAFS LC-40,Dragon demo flight C2+,525.0,LEO (ISS),NASA (COTS),Success,No attempt,0,28.562302,-80.577356
3,4,2012-10-08,0:35:00,F9 v1.0 B0006,CCAFS LC-40,SpaceX CRS-1,500.0,LEO (ISS),NASA (CRS),Success,No attempt,0,28.562302,-80.577356
4,5,2013-03-01,15:10:00,F9 v1.0 B0007,CCAFS LC-40,SpaceX CRS-2,677.0,LEO (ISS),NASA (CRS),Success,No attempt,0,28.562302,-80.577356


In [2]:
# Identify coordinate columns
lat_col = next((c for c in df.columns if c.lower() in ["lat", "latitude"]), None)
lon_col = next((c for c in df.columns if c.lower() in ["long", "lon", "longitude"]), None)
site_col = next((c for c in df.columns if "launch" in c.lower() and "site" in c.lower()), None)
class_col = next((c for c in df.columns if c.lower() == "class"), None)
print("Detected:", lat_col, lon_col, site_col, class_col)

Detected: Lat Long Launch Site class


In [3]:
# Build the map
if lat_col and lon_col:
    clean = df.dropna(subset=[lat_col, lon_col]).copy()
    center = [clean[lat_col].mean(), clean[lon_col].mean()]
    launch_map = folium.Map(location=center, zoom_start=4)
    cluster = MarkerCluster().add_to(launch_map)
    for _, row in clean.iterrows():
        success = int(row[class_col]) if class_col and pd.notna(row[class_col]) else None
        color = "green" if success == 1 else ("red" if success == 0 else "blue")
        popup = str(row[site_col]) if site_col else "Launch"
        folium.Marker(
            location=[row[lat_col], row[lon_col]],
            popup=popup,
            icon=folium.Icon(color=color, icon="info-sign")
        ).add_to(cluster)
    display(launch_map)
    launch_map.save("spacex_launch_map.html")
    print("Saved spacex_launch_map.html")
else:
    print("Could not detect latitude/longitude columns. Inspect the column list and set lat_col/lon_col manually.")

Saved spacex_launch_map.html


## Screenshot for the presentation
Run the map cell, zoom to show the launch locations, and capture a screenshot. The HTML file can also be committed to GitHub.